# Ligand-Receptor Motion: GPCRmd MLX NVT

This notebook consumes one active artifact only: the Slice 9 GPCRmd MLX short-range prototype proof trajectory saved by `run_gpcrmd_mlx`. It visualizes/analyzes the saved MLX output and does not run MD.


## Scope

The active result is the MLX-generated `run_gpcrmd_mlx` trajectory for the selected GPCRmd target. GPCRmd reference trajectory files are comparison context only. If the saved trajectory is missing, not `engine=mlx_atomistic`, not `workflow=run_gpcrmd_mlx`, or otherwise blocked, the notebook displays the blocker JSON and stops before trajectory visualization.


In [1]:
import importlib
import os
import sys
from dataclasses import replace
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "helpers").exists():
    NOTEBOOK_DIR = Path("notebooks/ligand-receptor-motion").resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import helpers.mlx_real_md as mlx_real_md  # noqa: E402
import helpers.motion_analysis as motion_analysis  # noqa: E402
import helpers.visualization as visualization  # noqa: E402

# Pick up local source edits without restarting the notebook kernel.
mlx_real_md = importlib.reload(mlx_real_md)
motion_analysis = importlib.reload(motion_analysis)
visualization = importlib.reload(visualization)

ensure_gpcrmd_mlx_bundle = mlx_real_md.ensure_gpcrmd_mlx_bundle
analysis_tables = motion_analysis.analysis_tables
ligand_com_displacement = motion_analysis.ligand_com_displacement
trajectory_quality_report = motion_analysis.trajectory_quality_report
make_ligand_motion_figure = visualization.make_ligand_motion_figure
playback_table = visualization.playback_table

TARGET_ID = os.environ.get("GPCRMD_MLX_TARGET", "gpcrmd-729-beta1-5f8u-cyanopindolol")
DATASET_NAME = os.environ.get("GPCRMD_MLX_DATASET", "729-proof")
GPCRMD_MLX_STEPS = int(os.environ.get("GPCRMD_MLX_STEPS", "10"))
DATA_DIR = NOTEBOOK_DIR / "data/gpcrmd-mlx" / DATASET_NAME
VIEW_STRIDE = int(os.environ.get("GPCRMD_MLX_VIEW_STRIDE", "1"))


## MLX Artifact Status And Runtime

This cell only loads the saved Slice 9 artifact. To regenerate it, use the producer command outside this notebook:

```bash
uv run atomistic-prep run-gpcrmd-mlx \
  --target gpcrmd-729-beta1-5f8u-cyanopindolol \
  --cache /path/to/gpcrmd-cache-or-manifest \
  --out notebooks/ligand-receptor-motion/data/gpcrmd-mlx/729-proof \
  --steps ${GPCRMD_MLX_STEPS:-10} --dt 0.0005 --sample-interval 1 \
  --diagnostic-interval ${GPCRMD_MLX_STEPS:-10}
```


In [2]:
bundle = ensure_gpcrmd_mlx_bundle(
    out_dir=DATA_DIR,
    target_id=TARGET_ID,
    steps=GPCRMD_MLX_STEPS,
    sample_interval=1,
    dt=0.0005,
    minimize_steps=0,
    equilibration_steps=0,
    diagnostic_interval=GPCRMD_MLX_STEPS,
)

status_row = {
    "prepared_dir": str(bundle.prepared_dir),
    "trajectory_path": str(bundle.trajectory_path),
    "report_path": str(bundle.report_path),
    "status": bundle.run_report.get("status"),
    "target_id": bundle.run_report.get("target_id"),
    "dynamics_id": bundle.run_report.get("dynamics_id"),
    "engine": bundle.metadata.get("engine"),
    "source": bundle.metadata.get("source"),
    "workflow": bundle.metadata.get("workflow"),
    "trajectory_written": bundle.run_report.get("trajectory_written"),
    "blockers": "; ".join(bundle.blockers),
}
display(pd.DataFrame([status_row]))

if bundle.processed_trajectory is None:
    display(Markdown("**GPCRmd MLX blocker JSON**"))
    display(Markdown(f"```json\n{bundle.blocker_json()}\n```"))
    raise RuntimeError("GPCRmd MLX trajectory is blocked; see blocker JSON above.")

trajectory = bundle.processed_trajectory
metadata = bundle.metadata

display(
    pd.DataFrame(
        [
            {
                "kind": trajectory.source.get("kind"),
                "engine": trajectory.source.get("engine"),
                "workflow": trajectory.source.get("workflow"),
                "target_id": trajectory.source.get("target_id"),
                "dynamics_id": trajectory.source.get("dynamics_id"),
                "source": trajectory.source.get("source"),
                "reference_role": trajectory.source.get("reference_role"),
                "electrostatics_route": metadata.get("electrostatics_route"),
                "electrostatics_model": trajectory.source.get("electrostatics_model"),
                "electrostatics_production_ready": metadata.get("electrostatics_production_ready"),
                "ensemble": trajectory.source.get("ensemble"),
                "proof_mode": trajectory.source.get("proof_mode"),
                "barostat_status": trajectory.source.get("barostat_status"),
                "simulated_time_ps": metadata.get("simulated_time_ps"),
                "elapsed_wall_seconds": metadata.get("elapsed_wall_seconds"),
                "integration_steps_per_s": metadata.get("integration_steps_per_second"),
            }
        ]
    )
)
display(playback_table(trajectory))

quality_report = trajectory_quality_report(
    trajectory,
    max_constraint_error_A=float(bundle.diagnostics["constraint_max_error_A"].max()),
)
quality_table = pd.DataFrame([{k: v for k, v in quality_report.items() if k != "warnings"}])
display(Markdown("**Visualization Quality Report**"))
display(quality_table)
for warning in quality_report["warnings"]:
    display(Markdown(f"- {warning}"))

for warning in metadata.get("warnings", []):
    display(Markdown(f"- {warning}"))


,prepared_dir,trajectory_path,report_path,status,target_id,dynamics_id,engine,source,workflow,trajectory_written,blockers
0,/Users/ac/dev/ai/science/mlx-atomistic/noteboo...,/Users/ac/dev/ai/science/mlx-atomistic/noteboo...,/Users/ac/dev/ai/science/mlx-atomistic/noteboo...,ran,gpcrmd-729-beta1-5f8u-cyanopindolol,729,mlx_atomistic,mlx_atomistic,run_gpcrmd_mlx,True,


,kind,engine,workflow,target_id,dynamics_id,source,reference_role,electrostatics_route,electrostatics_model,electrostatics_production_ready,ensemble,proof_mode,barostat_status,simulated_time_ps,elapsed_wall_seconds,integration_steps_per_s
0,gpcrmd_mlx_nvt,mlx_atomistic,run_gpcrmd_mlx,gpcrmd-729-beta1-5f8u-cyanopindolol,729,mlx_atomistic,comparison_only,short-range-prototype,short_range_electrostatics_prototype,False,NVT,short_nvt,not_required_for_nvt_proof,0.005,20.863781,0.4793


,source,dataset_id,frames,atoms,ligand_atoms,receptor_atoms,replicas,selected_replica,gpu_visible_atoms,display,first_time_ps,last_time_ps,ps_per_frame,aggregate_steps_per_s
0,gpcrmd_mlx_nvt,,11,92001,43,5195,1,0,92001,PBC-corrected,0.0,0.005,0.0005,None


**Visualization Quality Report**

,pbc_corrected_display,cell_lengths_A,raw_ligand_com_max_step_A,unwrapped_ligand_com_max_step_A,raw_large_step_count,raw_pbc_jump_count,pbc_corrected_jump_count,final_unwrapped_ligand_com_displacement_A,max_unwrapped_ligand_com_displacement_A,contact_count_delta,max_constraint_error_A
0,True,"[87.17031860351562, 87.15242004394531, 118.580...",2.025955,0.001552,0,0,0,0.012925,0.012925,38,0.099482


- Constraint error is high; inspect MD stability before interpreting motion.

- Imported with ParmEd as a parser only. MLX generates trajectories.

- Imported from GPCRmd cached topology/parameter files. No external MD engine was run.

## Trajectory Viewer

The animated ligand pose is the stored MLX trajectory shown in a PBC-corrected display frame. The viewer defaults to ligand, receptor pocket atoms, nearby waters/ions, and selected membrane context rather than all solvent/lipids. The transparent initial pose is static; the orange trail only shows progress up to the active frame. Full COM path, net displacement axis, and residue labels are available from the legend but hidden by default. For browser responsiveness, the viewer may display a strided subset of frames; analysis tables use the full saved trajectory.


In [3]:
if VIEW_STRIDE > 1:
    viewer_trajectory = replace(
        trajectory,
        positions=trajectory.positions[::VIEW_STRIDE],
        time_ps=trajectory.time_ps[::VIEW_STRIDE],
        source={**trajectory.source, "viewer_stride": VIEW_STRIDE},
    )
else:
    viewer_trajectory = trajectory

title = (
    "MLX GPCRmd short-range prototype proof, PBC-corrected display "
    f"({viewer_trajectory.frame_count} displayed frames)"
)
figure = make_ligand_motion_figure(
    viewer_trajectory,
    title=title,
    frame_interval_ms=80,
)
figure.show()


## Dynamics Diagnostics


In [ ]:
diagnostics = bundle.diagnostics
display(diagnostics.head())
display(diagnostics.tail())

energy_columns = [
    column
    for column in [
        "potential_energy_kJ_mol",
        "kinetic_energy_kJ_mol",
        "total_energy_kJ_mol",
    ]
    if column in diagnostics
]
energy_figure = px.line(
    diagnostics,
    x="time_ps",
    y=energy_columns,
    title="MLX energy diagnostics",
    labels={"value": "kJ/mol", "variable": "term"},
)
energy_figure.show()

health_columns = [
    column
    for column in ["temperature_K", "pressure", "constraint_max_error_A"]
    if column in diagnostics
]
health_figure = px.line(
    diagnostics,
    x="time_ps",
    y=health_columns,
    title="Temperature, pressure, and constraint diagnostics",
)
health_figure.show()


## Interaction Analysis


In [ ]:
tables = analysis_tables(trajectory)
display(tables["summary"])
display(tables["frames"].head())
display(tables["frames"].tail())
display(tables["contact_occupancy"].head(15))
display(tables["closest_residues"].head(20))

hbond_table = tables["hydrogen_bonds"]
if bool(hbond_table["available"].any()):
    display(hbond_table.head())
    display(hbond_table.tail())
else:
    display(Markdown(f"**Hydrogen bonds not available.** {hbond_table['note'].iloc[0]}"))

displacement = ligand_com_displacement(trajectory)
summary = pd.DataFrame(
    [
        {
            "final_ligand_com_displacement_A": float(displacement[-1]),
            "max_ligand_com_displacement_A": float(displacement.max()),
            "frames": trajectory.frame_count,
            "last_time_ps": float(trajectory.time_ps[-1]),
        }
    ]
)
display(summary)
